# Train the notes model (QLoRA)

Fine-tunes Llama 3.2 3B Instruct on the 150 dental notes with QLoRA (Hugging Face PEFT + Transformers).

Before running:
1. Runtime > Change runtime type > T4 GPU
2. Upload `notes_train.jsonl` (the v2 train split) when the upload cell runs
3. Llama 3.2 is gated: accept the license at huggingface.co/meta-llama/Llama-3.2-3B-Instruct and paste your HF token when asked

If a pinned version conflicts with Colab, bump it and re-run the install cell.

In [ ]:
!nvidia-smi
import torch
print("cuda available:", torch.cuda.is_available())

In [ ]:
!pip install -q transformers==4.46.2 peft==0.13.2 trl==0.12.1 bitsandbytes==0.45.5 accelerate==1.1.1 datasets==3.1.0

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick notes_train.jsonl

In [ ]:
import json
from datasets import Dataset
from transformers import AutoTokenizer

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

# Must match the repo Modelfile SYSTEM prompt (schema v2) exactly. Training and
# inference have to use the same system prompt or the model learns the wrong
# conditioning and drops fields like next_appointment at inference time.
SYSTEM_PROMPT = """You read one messy English dental note and output ONLY a JSON object, nothing else.

The JSON must use exactly these keys:
- patient_name: string
- codice_fiscale: string
- phone: string or null
- visit_date: date as YYYY-MM-DD or null
- procedures: list of strings
- invoices: list of objects, each with "description" (string) and "amount" (number)
- clinical_notes: string
- next_appointment: string or null

Rules:
- Use ONLY information present in the note. Never invent a value. If something is not in the note, use null for phone, visit_date and next_appointment, and an empty list or empty string for the others.
- Keep procedure shorthand exactly as written. Do NOT expand it: keep tokens like "rct", "ext", "comp", "perio", "opg", "caries", "prophy", "scaling", "abx", "fu" verbatim. Mapping shorthand to standard codes happens later in the application layer.
- procedures: list every treatment or item mentioned, e.g. ["caries 14", "filling 21"].
- invoices: one object per paid item. "description" is the procedure only - strip "paid" and "for" and the amount. Example: "paid 60 for x-ray 43" -> {"amount": 60, "description": "x-ray 43"}. A follow-up such as "fu 2wk" is NOT an invoice - never create an invoice from it.
- next_appointment: the follow-up interval as a number of days followed by the letter d. Convert weeks and months: "fu 1wk" -> "7d", "fu 2wk" -> "14d", "fu 3wk" -> "21d", "fu 1mo" -> "30d". If the note has no follow-up, use null. Output the actual number, e.g. "14d", never the literal "Nd".
- Output valid JSON only. No commentary, no markdown code fences."""

rows = []
with open("notes_train.jsonl") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
print("loaded", len(rows), "examples")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_text(row):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["input"]},
        {"role": "assistant", "content": json.dumps(row["output"])},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

dataset = Dataset.from_list([{"text": to_text(r)} for r in rows])
print(dataset[0]["text"][:300])

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir="notes_lora_out",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    max_seq_length=1024,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset,
    peft_config=lora_config,
)
trainer.train()

In [ ]:
trainer.model.save_pretrained("notes_lora_adapter")
tokenizer.save_pretrained("notes_lora_adapter")

!zip -r notes_lora_adapter.zip notes_lora_adapter
from google.colab import files
files.download("notes_lora_adapter.zip")

## Export to GGUF for Ollama

Merge the LoRA adapter into the base model, convert to GGUF with llama.cpp, and quantize to `q4_k_m` so it fits a 16GB Mac. The final cell downloads `dental_notes.gguf` — put it in the repo root next to `Modelfile`, then run `ollama create dental-notes -f Modelfile`.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

# reload the base in fp16 — the training copy is 4-bit and can't be merged cleanly
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="cpu")
merged = PeftModel.from_pretrained(base, "notes_lora_adapter")
merged = merged.merge_and_unload()
merged.save_pretrained("notes_merged")
tokenizer.save_pretrained("notes_merged")
print("merged model saved to notes_merged")

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py notes_merged --outfile notes_f16.gguf --outtype f16

In [ ]:
# build llama-quantize and shrink to q4_k_m so the model fits the 16GB Mac
!cd llama.cpp && cmake -B build -DLLAMA_CURL=OFF && cmake --build build --config Release -j --target llama-quantize
!./llama.cpp/build/bin/llama-quantize notes_f16.gguf dental_notes.gguf q4_k_m

In [ ]:
from google.colab import files
files.download("dental_notes.gguf")